# Zero-shot HTR/CRNN baseline: Kraken + McCATMuS

Notebook uruchamia zero-shot inference na lokalnym Malaysian dysgraphia dataset oraz na `Teklia/IAM-line` z Hugging Face. Kod korzysta z repo `mittagessen/kraken` i modelu `McCATMuS` (`10.5281/zenodo.13788177`).

Rekomendowany kernel: `C:\\Magisterka\\ZeroShot\\.venv311_kraken\\Scripts\\python.exe`.

## Dlaczego Kraken

| Repo | Ocena pod ten eksperyment |
| --- | --- |
| `arthurflor23/handwritten-text-recognition` | HTR-oriented i ma IAM, ale wymaga Pythona `<3.14`, workflow MLflow i brak natychmiastowego publicznego baseline'u z polskim charsetem. |
| `Calamari-OCR/calamari` | Bardzo dobry CRNN/CTC OCR, ale TensorFlow i model zoo sa mniej wygodne dla tego lokalnego Windowsowego setupu. |
| `clovaai/deep-text-recognition-benchmark` | Najczystszy klasyczny CRNN benchmark, ale jest pod scene/word OCR, stare zaleznosci i angielski charset pretrained. Slaby kandydat pod polskie znaki i HTR. |
| `mittagessen/kraken` | Najlepszy balans: gotowy HTR/OCR, publiczne modele, API/CLI, Unicode, line OCR bez segmentacji oraz latwy fine-tuning przez `ketos train --load ... --resize union/new`. |

Uwaga: McCATMuS ma alfabet z lacinskimi znakami laczonymi, m.in. combining ogonek/acute, ale nie pokrywa idealnie wszystkich polskich glifow w zero-shot. Przy fine-tuningu trzeba uzyc `--resize union` albo `--resize new`, z transkrypcjami znormalizowanymi do wybranej formy Unicode.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"C:\Magisterka")
CRNN_DIR = PROJECT_ROOT / "CRNN"
sys.path.insert(0, str(CRNN_DIR))

print(sys.executable)
print(PROJECT_ROOT)

In [ ]:
from types import SimpleNamespace
import kraken_zero_shot_inference as zsi

# Full Malaysian dataset is small enough to run completely.
# IAM_LIMIT=200 keeps CPU runtime practical; set to None for full IAM test split.
args = SimpleNamespace(
    malaysian_limit=None,
    iam_limit=200,
    malaysian_label_mode="csv_rows",  # one sample = image_path + text_id + text from CSV
    model_path="",
    threads=1,
    log_every=25,
    no_auto_invert=False,
)

predictions, summary = zsi.run_experiment(args)
summary

In [ ]:
predictions.head(10)[[
    "dataset", "sample_id", "reference", "prediction", "cer", "wer", "inference_seconds"
]]

## Pliki wynikowe

Skrypt zapisuje:

- `CRNN/results/kraken_zero_shot_predictions.csv`
- `CRNN/results/kraken_zero_shot_summary.csv`
- `CRNN/results/kraken_zero_shot_metadata.json`
- `CRNN/results/kraken_zero_shot_metrics_raw.xlsx`
- `CRNN/manifests/malaysian_eval_manifest.csv`

Dodatkowo w repo jest `CRNN/build_metrics_workbook.mjs`, ktory buduje czytelniejszy skoroszyt `CRNN/results/kraken_zero_shot_metrics.xlsx` z dashboardem i notatkami diagnostycznymi. Malaysian jest domyslnie liczony w trybie `csv_rows`, czyli dokladnie wedlug wierszy CSV.

In [ ]:
from pathlib import Path
results_dir = PROJECT_ROOT / "CRNN" / "results"
for name in [
    "kraken_zero_shot_predictions.csv",
    "kraken_zero_shot_summary.csv",
    "kraken_zero_shot_metadata.json",
    "kraken_zero_shot_metrics_raw.xlsx",
    "kraken_zero_shot_metrics.xlsx",
]:
    p = results_dir / name
    print(f"{name}: {p.exists()} - {p}")